# Day I8 — 측정 시스템 개관과 선택: 질문이 장비를 고른다

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I8.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I8.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 2 — "몇 걸음을 재느냐"가 변수를 정한다

변동성(stride time CV, [Day I13](day-I13.md))처럼 **걸음 사이의 흔들림**을 보려면 걸음이 많아야 합니다. 아래 코드는 장비마다 한 번에 담기는 stride 수를 어림하고, stride 수가 적으면 CV 추정값이 얼마나 흔들리는지 가상 자료로 보여 줍니다.

In [ ]:
import numpy as np

# 1) 측정 구간이 몇 stride를 담는가 (stride length 1.3 m 가정)
stride_len = 1.3
for name, metres in [("10 m 보행로, 앞뒤 2 m 가감속 제외", 10 - 4),
                     ("GAITRite류 보행 매트 약 5 m", 5.0),
                     ("힘판 2장(발 한 번씩)", None)]:
    if metres is None:
        print(f"{name:30s}: 시행당 완전한 stride 0개(발자국 2개만 힘을 잰다)")
    else:
        print(f"{name:30s}: 시행당 완전한 stride 약 {int(metres // stride_len)}개")
print(f"{'IMU·깔창으로 2분 연속 보행':30s}: stride 약 {int(120 / 1.1)}개 (stride time 1.1 s 가정)")

# 2) stride 수가 적으면 CV(변동계수) 추정이 얼마나 흔들리나 (가상)
rng = np.random.default_rng(8)
true_mean, true_cv = 1.10, 2.0                    # 참 평균 1.10 s, 참 CV 2 %
for n in [5, 10, 30, 100]:
    cvs = []
    for _ in range(5000):
        s = rng.normal(true_mean, true_mean * true_cv / 100, n)
        cvs.append(100 * s.std(ddof=1) / s.mean())
    lo, hi = np.percentile(cvs, [5, 95])
    print(f"stride {n:3d}개로 잰 CV: 90%가 {lo:.2f}–{hi:.2f}% 안에 (참값 2.00%)")